In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_const_value_xml, build_const_techs_xml, write_text, twh_to_ej, xy, gw_to_twh
from pathlib import Path

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/gas_H2_blend_const_value.xml`
    * `/input/policy/korea-2035/power/gas_H2_blend_const_techs.xml`
    * `/input/policy/korea-2035/power/gas_const_value.xml`
    * `/input/policy/korea-2035/power/gas_const_techs.xml`
    * `/input/policy/korea-2035/power/gas_shutdown.xml`

# Gas

LNG electricity generation (TWh) is reported in the BPESD for 2023, 2030, and 2035. Generation for 2025 is derived from observed data covering the period from December 2024 to November 2025, based on Monthly Energy Statistics (MES).

From 2030 onward, the BPESD’s LNG generation figures are interpreted to include the gas contribution from hydrogen co-firing. Because hydrogen generation is classified as carbon-free in the BPESD, the gas contribution embedded in hydrogen co-firing is assumed to be equal to the hydrogen generation itself. Accordingly, hydrogen generation is subtracted from the reported LNG generation, and the adjusted values are implemented as the LNG generation ceiling.

In [2]:
dictCapTWh = {2020: 146.18, 2023: 157.7, 2025:164.5, 2030: 161, 2035: 101.1}

In [7]:
years_cap, values_cap = xy(dictCapTWh)

fig = go.Figure()

for name, x, y, dash in [
    ("Current Policies", years_cap, values_cap, None),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2020, 2025, 2030, 2035]
annotations = []
for yr in target_years:
    val = dictCapTWh.get(yr)
    if val is not None:
        annotations.append(go.layout.Annotation(
            x=yr, y=val,
            xanchor='center', yanchor='bottom',
            text=f"{val:.1f} TWh",
            showarrow=True, arrowhead=1, ax=0, ay=-20
        ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(
        title='Year',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        tickvals=[2020, 2025, 2030, 2035],  # custom tick positions
        range=[2018, 2036]                  # xrange
    ),
    yaxis=dict(
        title='TWh',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        range=[0, 200]                       # yrange
    ),
)

import plotly.io as pio
pio.write_image(fig, "../figure/gas.jpg", width=800, height=600, scale=3)
fig.show()

In [4]:
dictCapTWhH2 = {2020: 0, 2023: 0, 2025: 0, 2030: 15.5*0.469, 2035: 32.8*0.469}

In [5]:
years = [2020, 2025, 2030, 2035]

In [6]:
{y: dictCapTWh[y] - (dictCapTWhH2[y]) for y in years}

{2020: 146.18, 2025: 164.5, 2030: 153.7305, 2035: 85.71679999999999}

In [23]:
years = [2020, 2025, 2030, 2035]
values = {y: twh_to_ej_str(dictCapTWh[y] - (dictCapTWhH2[y])) for y in years}
policy_name = "Gas-Ceiling"
policy_type = "tax"
subsector_name = 'gas'
tech_names = ['gas (CC)', 'gas (steam/CT)']


min_price_values_by_year = {
    2020: -10000,
    2025: -10000,
    2030: 0,
    2035: 0,
}

In [24]:
xml_value = build_const_value_xml(
    values_by_year=values,
    policy_name=policy_name,
    policy_type=policy_type,
    # min_price=True,
    # min_price_values_by_year=min_price_values_by_year,
)

xml_techs = build_const_techs_xml(
    years=[2020, 2025, 2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name,
    policy_name=policy_name,
    tech_names=tech_names,
    policy_type=policy_type,
)

In [25]:
value_path = f"../../input/policy/korea-2035/power/gas_const_value.xml"
techs_path = f"../../input/policy/korea-2035/power/gas_const_techs.xml"

write_text(value_path, xml_value)
write_text(techs_path, xml_techs)

print("Wrote:", Path(value_path).expanduser())
print("Wrote:", Path(techs_path).expanduser())

Wrote: ../../input/policy/korea-2035/power/gas_const_value.xml
Wrote: ../../input/policy/korea-2035/power/gas_const_techs.xml
